# 03 - RAG Pipeline
Build and test the FAISS-based retrieval pipeline over the knowledge base.

In [1]:
import sys, os
os.chdir(os.path.abspath(".."))
sys.path.append(os.path.abspath(".."))

from src.document_processor import load_and_chunk_documents
from src.embedder import build_faiss_index

chunks = load_and_chunk_documents("../knowledge_base")
print(f"Loaded {len(chunks)} chunks from knowledge base")
chunks[0].page_content[:300]

Loaded 50 chunks from knowledge base


'How do I reset my online banking password?\nTo reset your password, go to the login page and select "Forgot Password." Enter your registered email or phone number, and we will send a verification code. Enter the code and choose a new password that is at least 8 characters with a mix of letters, numbe'

In [4]:
from pathlib import Path

models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

index_path = str(models_dir / "faiss_index.bin")
meta_path = str(models_dir / "chunks.pkl")

build_faiss_index(
    chunks,
    index_path=index_path,
    meta_path=meta_path
)
print("FAISS index built.")

FAISS index built.


In [5]:
# Test retrieval with sample queries
import faiss, pickle
from sentence_transformers import SentenceTransformer
import numpy as np

embed_model = SentenceTransformer("all-MiniLM-L6-v2")
index = faiss.read_index("../models/faiss_index.bin")
with open("../models/chunks.pkl", "rb") as f:
    texts = pickle.load(f)

test_queries = [
    "How do I dispute a charge on my bill?",
    "My card was stolen, what should I do?",
    "How do I reset my password?",
    "Why is my internet not working?",
    "How do I close my account?"
]

for q in test_queries:
    q_emb = embed_model.encode([q], normalize_embeddings=True).astype("float32")
    scores, idx = index.search(q_emb, 3)
    print(f"\nQuery: {q}")
    for s, i in zip(scores[0], idx[0]):
        print(f"  score={s:.3f} | {texts[i][:100]}...")


Query: How do I dispute a charge on my bill?
  score=0.822 | How do I dispute a charge on my bill?
If you notice an incorrect charge, go to Billing > Dispute a C...
  score=0.477 | My card was charged twice for the same transaction.
Duplicate charges are usually automatically reve...
  score=0.467 | Why was I charged a late fee?
A late fee is applied if your payment is not received by the due date ...

Query: My card was stolen, what should I do?
  score=0.733 | I think my card information was stolen via a skimming device.
If you suspect card skimming, freeze y...
  score=0.732 | My debit card is lost or stolen. What should I do?
Immediately go to Account > Cards > Report Lost/S...
  score=0.544 | How do I report a fraudulent transaction?
If you notice a transaction you do not recognize, go to Ac...

Query: How do I reset my password?
  score=0.681 | How do I reset my online banking password?
To reset your password, go to the login page and select "...
  score=0.505 | Why is my accoun

## Notes
- Chunk size: 512, overlap: 50.
- Embedding model: all-MiniLM-L6-v2 (384-dim).
- Similarity metric: cosine via IndexFlatIP with normalized embeddings.
- Manually inspect top-3 results above to confirm retrieval precision before relying on it in the full pipeline.